# Sieci neuronowe i Deep Learning
# Temat 7: Mini projekt

## Zadanie 7.1

Przeanalizować poniższy projekt analizy danych (budowa sieci do predykcji zużycia paliwa) i uzupełnić brakujące fragmenty (zaznaczone w kodzie przez ```#####```).

## Zadanie 7.2*

Potraktować zbiór testowy z powyższej analizy jako zbiór walidacyjny i na jakiego podstawie dobrać optymalne hiperparametry rozważanego modelu.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

from IPython.display import Image

## Predykcja zużycia paliwa przez samochód (w milach na galon: MPG)

Analiza przeprowadzona zostanie w oparciu o zbiór danych
*Auto MPG* (dane i opis: https://archive.ics.uci.edu/ml/datasets/auto+mpg).

Wczytanie danych:

In [ ]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

df = pd.read_csv(url, names=column_names,
                 na_values = "?", comment='\t',
                 sep=" ", skipinitialspace=True)

df.tail()

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
393,27.0,4,140.0,86.0,2790.0,15.6,82,1
394,44.0,4,97.0,52.0,2130.0,24.6,82,2
395,32.0,4,135.0,84.0,2295.0,11.6,82,1
396,28.0,4,120.0,79.0,2625.0,18.6,82,1
397,31.0,4,119.0,82.0,2720.0,19.4,82,1


Usuwamy wiersze z brakami:

In [ ]:
print(df.isna().sum())

df = df.dropna()
df = df.reset_index(drop=True)
df.tail()

MPG             0
Cylinders       0
Displacement    0
Horsepower      6
Weight          0
Acceleration    0
Model Year      0
Origin          0
dtype: int64


,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
387,27.0,4,140.0,86.0,2790.0,15.6,82,1
388,44.0,4,97.0,52.0,2130.0,24.6,82,2
389,32.0,4,135.0,84.0,2295.0,11.6,82,1
390,28.0,4,120.0,79.0,2625.0,18.6,82,1
391,31.0,4,119.0,82.0,2720.0,19.4,82,1


Tworzymy zbiór treningowy (80% całego zbioru danych) i zbiór testowy:

In [ ]:
import sklearn
import sklearn.model_selection


df_temp, df_test = sklearn.model_selection.train_test_split(df, train_size=0.6, random_state=1)
df_train, df_valid = sklearn.model_selection.train_test_split(df_temp, train_size=0.5, random_state=1)
train_stats = df_train.describe().transpose()
train_stats

,count,mean,std,min,25%,50%,75%,max
MPG,117.0,24.103419,7.637193,13.0,18.0,23.7,30.0,46.6
Cylinders,117.0,5.239316,1.674605,3.0,4.0,4.0,6.0,8.0
Displacement,117.0,184.149573,102.288033,70.0,98.0,140.0,250.0,455.0
Horsepower,117.0,100.273504,35.214766,46.0,72.0,90.0,112.0,225.0
Weight,117.0,2905.752137,809.025541,1613.0,2200.0,2711.0,3410.0,5140.0
Acceleration,117.0,15.768376,2.725602,8.5,14.1,15.5,17.4,24.8
Model Year,117.0,76.384615,3.777990,70.0,73.0,77.0,80.0,82.0
Origin,117.0,1.615385,0.839192,1.0,1.0,1.0,2.0,3.0


In [ ]:
numeric_column_names = ['Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration']

df_train_norm, df_test_norm, df_valid_norm, df_temp_norm = df_train.copy(), df_test.copy(), df_valid.copy(), df_temp.copy()

for col_name in numeric_column_names:
    mean = train_stats.loc[col_name, 'mean']
    std  = train_stats.loc[col_name, 'std']
    df_train_norm.loc[:, col_name] = (df_train_norm.loc[:, col_name] - mean)/std
    df_test_norm.loc[:, col_name] = (df_test_norm.loc[:, col_name] - mean)/std
    df_valid_norm.loc[:, col_name] = (df_valid_norm.loc[:, col_name] - mean)/std
    df_temp_norm.loc[:, col_name] = (df_temp_norm.loc[:, col_name] - mean)/std

df_train_norm.tail()

/tmp/ipykernel_6088/689597302.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.74006467  0.45424659 -0.74006467 -0.74006467  1.64855786  0.45424659
 -0.74006467 -0.74006467 -1.33722031 -1.33722031  1.64855786  0.45424659
 -0.14290904 -0.74006467  1.64855786 -0.74006467 -0.74006467  0.45424659
 -0.74006467 -0.74006467  1.64855786  1.64855786 -0.74006467  0.45424659
  1.64855786 -0.74006467 -0.74006467 -0.74006467 -0.74006467 -0.74006467
 -0.74006467 -0.74006467 -0.74006467 -0.74006467 -0.74006467 -0.74006467
  1.64855786  1.64855786  1.64855786 -0.74006467  1.64855786  0.45424659
  1.64855786 -0.74006467 -0.74006467 -0.74006467 -0.74006467 -1.33722031
 -0.74006467 -0.74006467  0.45424659 -0.74006467  0.45424659  1.64855786
 -0.74006467 -0.74006467 -0.74006467  1.64855786  1.64855786  0.45424659
 -0.74006467 -0.74006467  1.64855786  1.64855786 -0.74006467  1.64855786
 -0.74006467  0.45424659  0.45424659 -0.740

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
316,29.8,-0.740065,-0.490278,-0.291739,-0.240724,-0.098465,80,3
353,31.6,-0.740065,-0.627146,-0.746093,-0.334665,0.928831,81,3
40,14.0,1.648558,1.308564,1.412092,1.471212,-1.015694,71,1
75,18.0,-0.740065,-0.617370,0.332999,0.033680,-0.465356,72,2
314,19.1,0.454247,0.399367,-0.291739,0.587432,1.075588,80,1


Grupujemy informacje o roku modelowym w następujący sposób:
$$
\text{bucket} =
\begin{cases}
0 & \text{jeśli year} < 73, \\
1 & \text{jeśli } 73 \leq \text{year} < 76, \\
2 & \text{jeśli } 76 \leq \text{year} < 79, \\
3 & \text{jeśli year} \geq 79.
\end{cases}
$$

In [ ]:
boundaries = torch.tensor([73, 76, 79])

v = torch.tensor(df_train_norm['Model Year'].values)
df_train_norm['Model Year Bucketed'] = torch.bucketize(v, boundaries, right=True)

v = torch.tensor(df_test_norm['Model Year'].values)
df_test_norm['Model Year Bucketed'] = torch.bucketize(v, boundaries, right=True)


v = torch.tensor(df_valid_norm['Model Year'].values)
df_valid_norm['Model Year Bucketed'] = torch.bucketize(v, boundaries, right=True)

v = torch.tensor(df_temp_norm['Model Year'].values)
df_temp_norm['Model Year Bucketed'] = torch.bucketize(v, boundaries, right=True)

numeric_column_names.append('Model Year Bucketed')

Stosujemy one-hot-encoding do nieuporządkowanej zmiennej jakościowej Origin:

In [ ]:
from torch.nn.functional import one_hot


total_origin = len(set(df_train_norm['Origin']))

origin_encoded = one_hot(torch.from_numpy(df_train_norm['Origin'].values) % total_origin)
x_train_numeric = torch.tensor(df_train_norm[numeric_column_names].values)
x_train = torch.cat([x_train_numeric, origin_encoded], 1).float()

origin_encoded = one_hot(torch.from_numpy(df_test_norm['Origin'].values) % total_origin)
x_test_numeric = torch.tensor(df_test_norm[numeric_column_names].values)
x_test = torch.cat([x_test_numeric, origin_encoded], 1).float()

origin_encoded = one_hot(torch.from_numpy(df_valid_norm['Origin'].values) % total_origin)
x_valid_numeric = torch.tensor(df_valid_norm[numeric_column_names].values)
x_valid = torch.cat([x_valid_numeric, origin_encoded], 1).float()

origin_encoded = one_hot(torch.from_numpy(df_temp_norm['Origin'].values) % total_origin)
x_temp_numeric = torch.tensor(df_temp_norm[numeric_column_names].values)
x_temp = torch.cat([x_temp_numeric, origin_encoded], 1).float()


Tworzymy tensory z wartościami zmiennej zależnej MPG:

In [ ]:
y_train = torch.tensor(df_train_norm['MPG'].values).float()
y_test = torch.tensor(df_test_norm['MPG'].values).float()
y_valid = torch.tensor(df_valid_norm['MPG'].values).float()
y_temp = torch.tensor(df_temp_norm['MPG'].values).float()

---

Tworzymy DataLoader z `batch_size = 8` dla danych uczących (`shuffle=True`):

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_ds = TensorDataset(x_train, y_train)
batch_size = 32
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

Budujemy sieć typu MLP z dwiema ukrytymi warstwami (rozmiaru 8 oraz 4):

In [ ]:
hidden_units = [ 18, 9, 4]
input_size = x_train.shape[1]

all_layers = []
for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(hidden_units[-1], 1))

model = nn.Sequential(*all_layers)

model

Sequential(
  (0): Linear(in_features=9, out_features=18, bias=True)
  (1): ReLU()
  (2): Linear(in_features=18, out_features=9, bias=True)
  (3): ReLU()
  (4): Linear(in_features=9, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=1, bias=True)
)

Wybieramy funkcję straty MSE oraz SGD jako optymalizator (`lr=0.001` na początek):

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0015)

Uczymy model przez 240 epok i wyświetlamy stratę na zbiorze uczącym co 20 epok:

In [ ]:
num_epochs = 240
log_epochs = 20

for epoch in range(num_epochs):
    loss_hist_train = 0

    for x_batch, y_batch in train_dl:
        pred = model(x_batch)[:, 0]
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


        loss_hist_train += loss.item()
    if epoch % log_epochs==0:
        print(f'Epoch {epoch}  Loss {loss_hist_train/len(train_dl):.4f}')

Epoch 0  Loss 623.8064
Epoch 20  Loss 10.8164
Epoch 40  Loss 10.0790
Epoch 60  Loss 10.8175
Epoch 80  Loss 11.5014
Epoch 100  Loss 9.4059
Epoch 120  Loss 6.7673
Epoch 140  Loss 7.7518
Epoch 160  Loss 12.0294
Epoch 180  Loss 9.8675
Epoch 200  Loss 8.4248
Epoch 220  Loss 6.9839


Sprawdzamy jakość modelu (MSE oraz MAE) na zbiorze testowym:

In [ ]:
with torch.no_grad():
    pred = model(x_valid.float())[:, 0]
    loss = loss_fn(pred, y_valid)
    print(f'Test MSE: {loss.item():.4f}')
    print(f'Test MAE: {nn.L1Loss()(pred, y_valid).item():.4f}')

Test MSE: 7.6501
Test MAE: 2.1343


In [ ]:
model_final = nn.Sequential(*all_layers)

train_ds = TensorDataset(x_temp, y_temp)
batch_size = 32
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

for epoch in range(num_epochs):
    loss_hist_train = 0

    for x_batch, y_batch in train_dl:
        pred = model(x_batch)[:, 0]
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


        loss_hist_train += loss.item()
    if epoch % log_epochs==0:
        print(f'Epoch {epoch}  Loss {loss_hist_train/len(train_dl):.4f}')

Epoch 0  Loss 8.0896
Epoch 20  Loss 6.9668
Epoch 40  Loss 13.1154
Epoch 60  Loss 7.5865
Epoch 80  Loss 5.5477
Epoch 100  Loss 5.8271
Epoch 120  Loss 10.9682
Epoch 140  Loss 5.0592
Epoch 160  Loss 5.2572
Epoch 180  Loss 6.0015
Epoch 200  Loss 6.1876
Epoch 220  Loss 5.5786


In [ ]:
with torch.no_grad():
    pred = model(x_test.float())[:, 0]
    loss = loss_fn(pred, y_test)
    print(f'Test MSE: {loss.item():.4f}')
    print(f'Test MAE: {nn.L1Loss()(pred, y_test).item():.4f}')

Test MSE: 9.9305
Test MAE: 2.2372
